/omics/groups/OE0132/internal/metzj/projects/Multiome/CellRanger_Output/Exp4/ctr/outs/atac_fragments.tsv.gz

I have a list of roughly 100 genes. I want to select a random subset and for this subset look at the distribution of atac counts for this gene (which I have as a cell x counts matrix) around its locus, using different cell subsets. 

Additionally I can provide the cells and the cell subsets (bins along a known development trajectory). From the distributions of the atac counts of the 10 selected genes in the different cellular bins I want to construct a single plot like the one the attached R code constructs (correct me if this code cannot be used for this). 

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from scipy.ndimage import gaussian_filter1d
import scanpy as sc

In [2]:
python_data_dir = rf"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_testing\data"

In [3]:
FRAGMENTS = rf"{python_data_dir}\atac_fragments.tsv.gz"
ATAC_H5AD = rf"{python_data_dir}\all_atac_peaks_corr.h5ad"
RNA_H5AD = rf"{python_data_dir}\total_counts_raw.h5ad"
GTF = rf"{python_data_dir}\genes.gtf.gz"
BINS_CSV = rf"{python_data_dir}\cell_bins.csv"

N_GENES, N_BINS, WINDOW, SMOOTH = 2, 10, 100, 2
UP, DOWN = 2000, 100
SEED = 0

In [16]:
gene_data = pd.read_csv(rf"{python_data_dir}\pmtr_peak_sub\chrom_velo_all_genes.csv", index_col=0, header=0)
gene_subset = gene_data[(gene_data["monotonicity_r"] <= 0.1) 
          & (gene_data["monotonicity_a"] <= 0.1) 
          & (gene_data["rna_dir"] == 1)
          & (gene_data["atac_dir"] == -1)]
          #& (gene_data["rna_dir"] == gene_data["atac_dir"])]

GENE_LIST = gene_subset.index.tolist() # rf"{python_data_dir}\genes.txt"

In [15]:
bins = pd.read_csv(BINS_CSV, header=0, index_col=0)["bin"]
atac = sc.read_h5ad(ATAC_H5AD)
rna = sc.read_h5ad(RNA_H5AD)

rna.obs_names = [i[9:-2] for i in rna.obs_names.values]

bins = bins[bins.index.isin(atac.obs_names)]
rna = rna[bins.index.intersection(rna.obs_names)]
bin_of = bins.to_dict()
barcodes = set(bin_of)

depth = pd.Series(np.asarray(atac.X.sum(1)).ravel(), index=atac.obs_names)
depth = depth.reindex(bins.index).groupby(bins).sum().reindex(range(N_BINS))

first_chrom = str(pd.read_csv(FRAGMENTS, sep="\t", comment="#", header=None, nrows=1)[0].iloc[0])
prefix = "chr" if first_chrom.startswith("chr") else ""

gtf = pd.read_csv(GTF, sep="\t", comment="#", header=None, usecols=[0, 2, 3, 4, 6, 8],
                  names=["chrom", "feature", "start", "end", "strand", "attr"])
gtf = gtf[gtf.feature == "gene"].copy()
gtf["gene"] = gtf.attr.str.extract(r'gene_name "([^"]+)"')
gtf["chrom"] = prefix + gtf.chrom.astype(str).str.replace("^chr", "", regex=True)
genes = gtf.dropna(subset=["gene"]).drop_duplicates("gene").set_index("gene")

In [17]:
bins_raw = pd.read_csv(BINS_CSV, header=0, index_col=0)["bin"]
print("bins:", len(bins_raw), bins_raw.dtype, sorted(bins_raw.unique())[:12])
print("bins ids:", bins_raw.index[:3].tolist())
print("atac ids:", atac.obs_names[:3].tolist())
print("rna ids: ", rna.obs_names[:3].tolist())
print("overlap bins/atac:", bins_raw.index.isin(atac.obs_names).sum())
print("overlap bins/rna: ", bins_raw.index.isin(rna.obs_names).sum())

frag_head = pd.read_csv(FRAGMENTS, sep="\t", comment="#", header=None, nrows=5, usecols=[3])
print("fragment ids:", frag_head[3].tolist())

bins: 4238 int64 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
bins ids: ['AAACAGCCACTAAGTT', 'AAACAGCCAGCATGAG', 'AAACAGCCAGGCCATT']
atac ids: ['AAACAGCCACTAAGTT', 'AAACAGCCAGCATGAG', 'AAACAGCCAGGCCATT']
rna ids:  ['AAACAGCCACTAAGTT', 'AAACAGCCAGCATGAG', 'AAACAGCCAGGCCATT']
overlap bins/atac: 4238
overlap bins/rna:  4238
fragment ids: ['TACGGTTAGGGCTTTG-1', 'TTTCCGGGTGTGAGGA-1', 'CATCAATCATCTAGCA-1', 'AGGATCCGTCCCGGAA-1', 'CCCTCAGTCTTAGGGT-1']


In [14]:
rna.obs_names.values

array([], dtype=object)

In [32]:
def load_fragments(path, regions):
    by_chrom = {}
    for gene, (c, s, e) in regions.items():
        by_chrom.setdefault(c, []).append((gene, s, e))
    out = {gene: [] for gene in regions}
    
    reader = pd.read_csv(path, sep="\t", comment="#", header=None, usecols=[0, 1, 2, 3],
                         names=["chrom", "start", "end", "bc"], chunksize=10_000, nrows=10000000)
    for chunk in reader:
        chunk = chunk[chunk.chrom.isin(by_chrom) & chunk.bc.isin(barcodes)]
        for c, items in by_chrom.items():
            sub = chunk[chunk.chrom == c]
            for gene, s, e in items:
                out[gene].append(sub[(sub.end >= s) & (sub.start <= e)])
    return {gene: pd.concat(v) for gene, v in out.items()}


def coverage(frag, start, end):
    edges = np.arange(start, end + WINDOW, WINDOW)
    cov = np.zeros((N_BINS, len(edges) - 1))
    for b, grp in frag.assign(bin=frag.bc.map(bin_of)).groupby("bin"):
        cov[int(b)] = np.histogram(np.r_[grp.start, grp.end], bins=edges)[0]
    cov = cov / depth.values[:, None] * 1e6
    return edges[:-1] + WINDOW / 2, gaussian_filter1d(cov, SMOOTH, axis=1)


def expr_per_bin(gene):
    x = rna[:, gene].X
    x = x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()
    return pd.Series(x, index=rna.obs_names).groupby(bins.reindex(rna.obs_names)).mean().reindex(range(N_BINS))

In [33]:
cmap, norm = plt.get_cmap("viridis"), colors.Normalize(0, N_BINS - 1)
rng = np.random.default_rng(SEED)
candidates = GENE_LIST
selected = rng.choice(candidates, min(N_GENES, len(candidates)), replace=False)
print(selected)

['Pag1' 'Asap1']


In [34]:
regions = {}
for gene in selected:
    g = genes.loc[gene]
    left, right = (UP, DOWN) if g.strand == "+" else (DOWN, UP)
    regions[gene] = (g.chrom, max(int(g.start) - left, 0), int(g.end) + right)

In [35]:
regions

{'Pag1': ('chr3', 9687379, 9835679), 'Asap1': ('chr15', 64086757, 64384919)}

In [36]:
fragments = load_fragments(FRAGMENTS, regions)

In [38]:
fragments

{'Pag1': Empty DataFrame
 Columns: [chrom, start, end, bc]
 Index: [],
 'Asap1': Empty DataFrame
 Columns: [chrom, start, end, bc]
 Index: []}

In [39]:

for gene in selected:
    g = genes.loc[gene]
    chrom, start, end = regions[gene]
    x, cov = coverage(fragments[gene], start, end)
    expr = expr_per_bin(gene)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained",
                                   gridspec_kw={"width_ratios": [3, 1]})
    for b in range(N_BINS):
        ax1.plot(x, cov[b], color=cmap(norm(b)), lw=1.5)
    ax1.axvspan(g.start, g.end, color="grey", alpha=0.1)
    if g.strand == "-":
        ax1.invert_xaxis()
    ax1.set(title=f"{gene} ({g.strand})", xlabel=f"{chrom} position (bp)",
            ylabel="ATAC insertions per million")

    ax2.plot(range(N_BINS), expr.values, color="grey", lw=1, zorder=1)
    ax2.scatter(range(N_BINS), expr.values, c=range(N_BINS), cmap=cmap, norm=norm, s=50, zorder=2)
    ax2.set(xlabel="Cell bin", ylabel="Mean expression")

    fig.colorbar(cm.ScalarMappable(norm, cmap), ax=[ax1, ax2], label="Cell bin")
    #fig.savefig(f"{gene}_atac_rna.pdf")
    plt.show()
    #plt.close(fig)

: 

If your fragment barcodes carry a sample# prefix, strip it in load_fragments right after the chunk is read, e.g. chunk["bc"] = chunk.bc.str.split("#").str[-1], before the isin(barcodes) filter.